In [1]:
!pip install dash plotly pandas more-itertools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 30.4 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python
# coding: utf-8

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

# In Google Colab this import is used to display the Dash app inside the notebook
from google.colab import output

# Load the data using pandas
data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d51iMGfp_t0QpO30Lym-dw/automobile-sales.csv'
)

# Clean column names from possible leading/trailing spaces
data.columns = data.columns.str.strip()

# Initialize the Dash app
app = dash.Dash(__name__)
app.title = "Automobile Sales Statistics Dashboard"

# ---------------------------------------------------------------------------------
# TASK 2.2: Create dropdown menu options

dropdown_options = [
    {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
]

# List of years
# NOTE: requirement says year range is between 1980 and 2013
year_list = [i for i in range(1980, 2014, 1)]

# ---------------------------------------------------------------------------------
# Create the layout of the app

app.layout = html.Div([

    # TASK 2.1: Add title to the dashboard
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': '24px'
        }
    ),

    # TASK 2.2: Add first dropdown menu for report type
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=dropdown_options,
            value='Select Statistics',
            placeholder='Select a report type',
            style={
                'width': '80%',
                'padding': '3px',
                'font-size': '20px',
                'text-align-last': 'center'
            }
        )
    ]),

    # TASK 2.2: Add second dropdown menu for year selection
    html.Div([
        html.Label("Select Year:"),
        dcc.Dropdown(
            id='select-year',
            options=[{'label': i, 'value': i} for i in year_list],
            value=1980,
            placeholder='Select-year',
            style={
                'width': '80%',
                'padding': '3px',
                'font-size': '20px',
                'text-align-last': 'center'
            }
        )
    ]),

    # TASK 2.3: Add a division for output display
    html.Div([
        html.Div(
            id='output-container',
            className='chart-grid',
            style={
                'display': 'flex',
                'flexDirection': 'column'
            }
        )
    ])
])

# ---------------------------------------------------------------------------------
# TASK 2.4: Creating Callbacks

# Callback 1: Enable or disable year dropdown
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics', component_property='value')
)
def update_input_container(selected_statistics):
    """
    Enable year dropdown only when Yearly Statistics is selected.
    Disable it when Recession Period Statistics is selected.
    """
    if selected_statistics == 'Yearly Statistics':
        return False
    else:
        return True


# Callback 2: Update output container with plots
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [
        Input(component_id='dropdown-statistics', component_property='value'),
        Input(component_id='select-year', component_property='value')
    ]
)
def update_output_container(selected_statistics, input_year):

    # -----------------------------------------------------------------------------
    # TASK 2.5: Recession Report Statistics
    # -----------------------------------------------------------------------------
    if selected_statistics == 'Recession Period Statistics':

        # Filter the data for recession periods
        recession_data = data[data['Recession'] == 1]

        # Plot 1: Automobile sales fluctuation over recession period, year-wise
        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()

        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title='Average Automobile Sales fluctuation over Recession Period'
            )
        )

        # Plot 2: Average number of vehicles sold by vehicle type during recessions
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()

        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Number of Vehicles Sold by Vehicle Type during Recessions'
            )
        )

        # Plot 3: Pie chart for total expenditure share by vehicle type during recessions
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()

        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_rec,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Expenditure Share by Vehicle Type During Recessions'
            )
        )

        # Plot 4: Bar chart for the effect of unemployment rate on vehicle type and sales
        unemp_data = recession_data.groupby(
            ['unemployment_rate', 'Vehicle_Type']
        )['Automobile_Sales'].mean().reset_index()

        R_chart4 = dcc.Graph(
            figure=px.bar(
                unemp_data,
                x='unemployment_rate',
                y='Automobile_Sales',
                color='Vehicle_Type',
                labels={
                    'unemployment_rate': 'Unemployment Rate',
                    'Automobile_Sales': 'Average Automobile Sales'
                },
                title='Effect of Unemployment Rate on Vehicle Type and Sales'
            )
        )

        # Return four charts in 2 rows and 2 columns
        return [
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=R_chart1, style={'width': '50%'}),
                    html.Div(children=R_chart2, style={'width': '50%'})
                ],
                style={'display': 'flex'}
            ),
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=R_chart3, style={'width': '50%'}),
                    html.Div(children=R_chart4, style={'width': '50%'})
                ],
                style={'display': 'flex'}
            )
        ]

    # -----------------------------------------------------------------------------
    # TASK 2.6: Yearly Report Statistics
    # -----------------------------------------------------------------------------
    elif selected_statistics == 'Yearly Statistics' and input_year:

        # Filter the data for the selected year
        yearly_data = data[data['Year'] == input_year]

        # Plot 1: Yearly Automobile sales using line chart for the whole period
        yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()

        Y_chart1 = dcc.Graph(
            figure=px.line(
                yas,
                x='Year',
                y='Automobile_Sales',
                title='Yearly Automobile Sales'
            )
        )

        # Plot 2: Total Monthly Automobile sales using line chart for selected year
        mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()

        Y_chart2 = dcc.Graph(
            figure=px.line(
                mas,
                x='Month',
                y='Automobile_Sales',
                title='Total Monthly Automobile Sales'
            )
        )

        # Plot 3: Average number of vehicles sold by vehicle type in selected year
        avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()

        Y_chart3 = dcc.Graph(
            figure=px.bar(
                avr_vdata,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Vehicles Sold by Vehicle Type in the year {}'.format(input_year)
            )
        )

        # Plot 4: Total Advertisement Expenditure for each vehicle type in selected year
        exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()

        Y_chart4 = dcc.Graph(
            figure=px.pie(
                exp_data,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertisement Expenditure for Each Vehicle'
            )
        )

        # Return four charts in 2 rows and 2 columns
        return [
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=Y_chart1, style={'width': '50%'}),
                    html.Div(children=Y_chart2, style={'width': '50%'})
                ],
                style={'display': 'flex'}
            ),
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=Y_chart3, style={'width': '50%'}),
                    html.Div(children=Y_chart4, style={'width': '50%'})
                ],
                style={'display': 'flex'}
            )
        ]

    else:
        return html.Div(
            "Please select a report type.",
            style={
                'textAlign': 'center',
                'font-size': '18px',
                'marginTop': '20px'
            }
        )


# ---------------------------------------------------------------------------------
# Run the Dash app inside Google Colab

output.serve_kernel_port_as_iframe(8050, height=900)

app.run(
    host='0.0.0.0',
    port=8050,
    debug=False
)


<IPython.core.display.Javascript object>

Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [03/May/2026 11:37:39] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 11:37:39] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1777807313.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 11:37:39] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1777807313.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 11:37:39] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1777807313.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 11:37:39] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_1_0m1777807313.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 11:37:39] "GET /_da